# ЛР № 4. Витрина Airbnb Amsterdam

Аглушевич Павел, Букатка Илона, Притужалов Илья, ПИН-241.

**Основной вопрос:** Как различаются заявленные суточные цены и календарная доступность
между районами при сравнении одинаковых типов жилья?

Продолжаем [Project Brief](../docs/project_brief.md) и
[решения ЛР № 3](../docs/data_quality_report.md). Готовим данные для выбранного
основного вопроса. Срезы: район и тип жилья. Будущие показатели: количество
объявлений, медианная заявленная цена в евро за ночь и медианная календарная
доступность в днях. В этой лабораторной выполнены сборка витрины
и проверки ее пригодности для выбранного вопроса. Модель и происхождение полей описаны в [docs/mart.md](../docs/mart.md).

## 1. Параметры и вход

Логическая дата берется из пути подготовленного `staging`, а не из даты запуска.
`source_ingested_on=2026-10-23` обозначает загрузку исходных данных. Дата версии
источника - 2026-06-15, фактический сбор объявлений и календарей проходил
15-24 июня 2026. Дата загрузки не используется как дата сбора.

Ноутбук запускается из корня репозитория или каталога `notebooks` с ядром
проектного `.venv`. Стенд должен работать, настройки S3 читаются из `.env`.
Используется подключение ЛР № 3. Для работы на другом компьютере проверенная
копия `docs/evidence/staging.parquet` перенесена в локальное S3-хранилище.
Ее контрольная сумма до и после переноса совпала с зафиксированной в ЛР № 3.
Далее вход читается непосредственно из S3.

Отсутствующие цены сохраняются с флагами и ограничивают ценовой анализ.
Неизвестный календарный признак ограничивает расчет основной медианы доступности.
Малые группы и различия в датах сбора учитываются при дальнейшем сравнении.

In [1]:
STAGING_PATH = (
    "s3://staging/airbnb_amsterdam_listings/"
    "source_ingested_on=2026-10-23/staging.parquet"
)

MART_PATH = (
    "s3://mart/airbnb_amsterdam_listings/"
    "source_ingested_on=2026-10-23/listings.parquet"
)

print("Вход:", STAGING_PATH)
print("Выход:", MART_PATH)

Вход: s3://staging/airbnb_amsterdam_listings/source_ingested_on=2026-10-23/staging.parquet
Выход: s3://mart/airbnb_amsterdam_listings/source_ingested_on=2026-10-23/listings.parquet


In [2]:
import os
import sys
from html import escape
from pathlib import Path
from urllib.parse import urlsplit

import boto3
import duckdb
from dotenv import load_dotenv
from IPython.display import HTML, display

ROOT = next(
    p
    for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "pyproject.toml").exists() and (p / "src").is_dir()
)
load_dotenv(ROOT / ".env", override=False)
sys.path.insert(0, str(ROOT / "src"))

from common.s3 import configure_s3  # noqa: E402


def sql_literal(value):
    return "'" + str(value).replace("'", "''") + "'"


def show_table(columns, rows):
    header = "".join(f"<th>{escape(str(value))}</th>" for value in columns)
    body = "".join(
        "<tr>" + "".join(f"<td>{escape(str(value))}</td>" for value in row) + "</tr>"
        for row in rows
    )
    display(HTML(f"<table><thead><tr>{header}</tr></thead><tbody>{body}</tbody></table>"))


def show_sql(query, columns):
    rows = con.execute(query).fetchall()
    show_table(columns, rows)
    return rows


check_results = []


def check(label, actual, expected):
    passed = actual == expected
    check_results.append((label, actual, expected, passed))
    assert passed, f"{label}: получено {actual!r}, ожидалось {expected!r}"


con = duckdb.connect()
configure_s3(con)

endpoint = os.environ["S3_ENDPOINT"]
s3 = boto3.client(
    "s3",
    endpoint_url=endpoint if "://" in endpoint else "http://" + endpoint,
    aws_access_key_id=os.environ["S3_ACCESS_KEY"],
    aws_secret_access_key=os.environ["S3_SECRET_KEY"],
    region_name="us-east-1",
)
staging_url = urlsplit(STAGING_PATH)
mart_url = urlsplit(MART_PATH)
assert staging_url.scheme == mart_url.scheme == "s3"
assert staging_url.netloc == "staging"
assert mart_url.netloc == "mart"
assert mart_url.path == staging_url.path.replace(
    "/staging.parquet", "/listings.parquet"
)
staging_key = staging_url.path.lstrip("/")
mart_key = mart_url.path.lstrip("/")
source_before = s3.head_object(Bucket="staging", Key=staging_key)

In [3]:
con.execute(f"""
    CREATE OR REPLACE TEMP VIEW staging AS
    SELECT * FROM read_parquet({sql_literal(STAGING_PATH)}, hive_partitioning=false)
""")
source_schema = {row[0]: row[1] for row in con.execute("DESCRIBE staging").fetchall()}
required_fields = {
    "id", "neighbourhood_cleansed", "room_type", "price_eur", "availability_365",
    "has_availability", "last_scraped", "calendar_last_scraped", "flag_price_missing",
    "flag_price_invalid", "flag_calendar_unknown", "flag_calendar_inconsistent",
}
check("Наличие нужных полей staging", required_fields - source_schema.keys(), set())

staging_count = con.execute("SELECT count(*) FROM staging").fetchone()[0]


input_checks = show_sql("""
    SELECT
        count(*) AS input_rows,
        count(*) FILTER (
            WHERE id IS NULL OR trim(id) = ''
        ) AS missing_id,
        count(id) - count(DISTINCT id) AS duplicate_id,
        count(*) FILTER (
            WHERE neighbourhood_cleansed IS NULL
               OR trim(neighbourhood_cleansed) = ''
        ) AS missing_district,
        count(*) FILTER (
            WHERE room_type IS NULL OR trim(room_type) = ''
        ) AS missing_room_type,
        count(*) FILTER (
            WHERE try_cast(last_scraped AS DATE) IS NULL
        ) AS invalid_listing_date,
        count(*) FILTER (
            WHERE try_cast(calendar_last_scraped AS DATE) IS NULL
        ) AS invalid_calendar_date
    FROM staging
""", [
    'Строк staging',
    'Пустых id',
    'Повторов id',
    'Без района',
    'Без типа жилья',
    'Некорректных дат объявления',
    'Некорректных дат календаря',
])[0]

for label, value in zip(
    ["Пропуски ключа на входе", "Повторы ключа на входе", "Пропуски района на входе",
     "Пропуски типа жилья на входе", "Некорректные даты объявления на входе",
     "Некорректные даты календаря на входе"],
    input_checks[1:],
    strict=True,
):
    check(label, value, 0)

show_table(["Нужное поле staging", "Тип"], [
    (name, source_schema[name]) for name in sorted(required_fields)
])

show_sql("""
    SELECT
        count(*) FILTER (WHERE flag_price_missing) AS missing_prices,
        count(*) FILTER (WHERE flag_price_invalid) AS invalid_prices,
        count(*) FILTER (WHERE flag_calendar_unknown) AS unknown_calendars,
        count(*) FILTER (
            WHERE flag_calendar_inconsistent
        ) AS inconsistent_calendars
    FROM staging
""", [
    'Без цены с флагом',
    'Некорректных цен с флагом',
    'Неизвестный календарь',
    'Календарное несоответствие',
]);

Строк staging,Пустых id,Повторов id,Без района,Без типа жилья,Некорректных дат объявления,Некорректных дат календаря
10369,0,0,0,0,0,0


Нужное поле staging,Тип
availability_365,INTEGER
calendar_last_scraped,VARCHAR
flag_calendar_inconsistent,BOOLEAN
flag_calendar_unknown,BOOLEAN
flag_price_invalid,BOOLEAN
flag_price_missing,BOOLEAN
has_availability,BOOLEAN
id,VARCHAR
last_scraped,VARCHAR
neighbourhood_cleansed,VARCHAR


Без цены с флагом,Некорректных цен с флагом,Неизвестный календарь,Календарное несоответствие
3992,0,136,151


## 2. Гранулярность и модель

**Одна строка `listings` представляет одно объявление из выбранного снимка `staging`.**
Ключ `id` идентифицирует объявление внутри снимка. В ЛР № 3 пропусков и повторов
`id` не обнаружено. При сборке сохраняются все строки, дедупликация и `DISTINCT`
не применяются. Заполненность и уникальность ключа проверяются повторно.

Выбираем одну широкую таблицу: район, тип жилья, цена, календарная доступность,
признак наличия календаря, даты сбора и флаги качества находятся в одной строке.
Все необходимые атрибуты уже есть в `staging`, отдельные измерения и JOIN не требуются.

| Будущая метрика | Единица и правило расчета | Правило объединения срезов |
| --- | --- | --- |
| Количество объявлений | `count(DISTINCT id)` всех сохраненных объявлений, независимо от наличия цены и календаря | Количества складываются для непересекающихся групп одного снимка |
| Медианная заявленная суточная цена | `median(price_eur)` в евро за ночь для положительных заполненных цен без флагов отсутствующей или некорректной цены | Пересчитать из значений нужного среза, не складывать и не усреднять медианы групп |
| Медианная календарная доступность | `median(availability_365)` в днях для значений от 0 до 365 при `has_availability = true` | Пересчитать из значений нужного среза, не складывать и не усреднять медианы групп |

Цена используется для сравнения предложений. Сумма цен не является выручкой,
а сумма доступных дней разных объявлений не является длительностью периода.
Разные снимки могут содержать одни и те же объявления, поэтому их количества
нельзя складывать без учета пересечений.

Срезы сравнения - район и тип жилья. Для каждой будущей медианы требуется
показывать число пригодных записей. Группы с менее чем 20 пригодными наблюдениями
не включаются в рейтинг районов по соответствующей метрике. Для каждой группы
также показывается доля отсутствующих или некорректных цен согласно Project Brief.
Эти правила относятся к будущему расчету и не фильтруют строки при сборке.

Даты сбора объявления и календаря сохраняются отдельно. Доступность описывает
следующие 365 дней относительно сбора календаря. Используется один снимок,
история изменений объявлений отсутствует.

### Общее правило для пропусков цены и календарного признака

Объявления с отсутствующей исходной ценой сохранены в ЛР № 3 с соответствующим
флагом. Они остаются для подсчета предложений и анализа доступности.
Нулевая доступность допустима и участвует в основной медиане при
`has_availability = true`. Значения `false` и `NULL` не участвуют в этой медиане,
но соответствующие строки остаются в витрине.

Все строки staging включаются в витрину. Значения мер и четыре флага качества
переносятся без изменения. Ограничения цены и календарного признака применяются
отдельно при расчете соответствующих показателей, а не при сборке витрины.
Диагностический флаг календарного несоответствия сохраняется вместе со значениями.

## 3. SQL-преобразование

Витрина содержит 12 полей, необходимых для выбранного вопроса.
`neighbourhood_cleansed` переименовывается в `district`.
Даты сбора преобразуются из VARCHAR в DATE: перед преобразованием
проверено отсутствие пропущенных и нераспознаваемых дат.

Ключ, меры, календарный признак и четыре флага качества
переносятся без изменения значений. Исходные текстовые копии полей
и технический номер строки для выбранного вопроса не требуются.

Фильтрация, DISTINCT, группировки и JOIN при сборке не применяются.
Ожидается сохранение всех строк и исходной гранулярности.

### Сборка широкой таблицы

In [4]:
con.execute("""
    CREATE OR REPLACE TEMP TABLE listings AS
    SELECT
        id,
        neighbourhood_cleansed AS district,
        room_type,
        price_eur,
        availability_365,
        has_availability,
        CAST(last_scraped AS DATE) AS last_scraped,
        CAST(calendar_last_scraped AS DATE) AS calendar_last_scraped,
        flag_price_missing,
        flag_price_invalid,
        flag_calendar_unknown,
        flag_calendar_inconsistent
    FROM staging
""")

balance = show_sql("""
    SELECT
        (SELECT count(*) FROM staging) AS staging_rows,
        (SELECT count(*) FROM listings) AS mart_rows,
        (SELECT count(*) FROM staging)
            - (SELECT count(*) FROM listings) AS excluded_rows
""", ['Строк staging', 'Подготовлено', 'Исключено'])[0]

check("Баланс при преобразовании", balance[0], balance[1] + balance[2])
check("Сохранение строк при преобразовании", balance[1], staging_count)
check("Исключенные строки", balance[2], 0)

show_sql("""
    SELECT
        id,
        district,
        room_type,
        price_eur,
        availability_365,
        has_availability,
        last_scraped,
        calendar_last_scraped
    FROM listings
    ORDER BY id
    LIMIT 5
""", [
    'id',
    'Район',
    'Тип жилья',
    'Цена, евро за ночь',
    'Доступность, дней',
    'Наличие календаря',
    'Дата сбора объявления',
    'Дата сбора календаря',
]);

Строк staging,Подготовлено,Исключено
10369,10369,0


id,Район,Тип жилья,"Цена, евро за ночь","Доступность, дней",Наличие календаря,Дата сбора объявления,Дата сбора календаря
1000033344100120272,De Baarsjes - Oud-West,Entire home/apt,None,0,True,2026-06-24,2026-06-24
1000066837570466403,Noord-West,Entire home/apt,None,0,True,2026-06-24,2026-06-24
1000084611260726063,Bos en Lommer,Entire home/apt,None,0,None,2026-06-24,2026-06-24
1000137169358231636,Slotervaart,Entire home/apt,None,0,True,2026-06-24,2026-06-24
1000446069572409663,Centrum-Oost,Entire home/apt,228.33,31,True,2026-06-16,2026-06-16


## 4. Запись и повторное чтение S3

Удаляем только целевой объект `listings.parquet`
и записываем его заново. Повторный запуск заменяет результат, не добавляя
копий строк. Если запись прервется после удаления, объект потребуется
восстановить повторным выполнением ноутбука.

После записи открываем новое соединение DuckDB и читаем именно объект S3.
Проверки выполняются в новом соединении. Входной staging также читается заново
для независимого сопоставления сохраненной витрины с ожидаемым преобразованием.
Автоматическое добавление полей из каталогов отключено, проверяется физическая
схема файла. Логическая дата остается в пути.

In [5]:
s3.head_bucket(Bucket="mart")
s3.delete_object(Bucket="mart", Key=mart_key)
con.execute(f"""
    COPY listings TO {sql_literal(MART_PATH)}
    (FORMAT PARQUET, COMPRESSION ZSTD)
""")

writer = con
con = duckdb.connect()
configure_s3(con)
con.execute(f"""
    CREATE OR REPLACE VIEW staging AS
    SELECT * FROM read_parquet({sql_literal(STAGING_PATH)}, hive_partitioning=false)
""")
con.execute(f"""
    CREATE OR REPLACE VIEW published_listings AS
    SELECT * FROM read_parquet({sql_literal(MART_PATH)}, hive_partitioning=false)
""")


con.execute("""
    CREATE OR REPLACE VIEW listings AS
    SELECT
        id,
        neighbourhood_cleansed AS district,
        room_type,
        price_eur,
        availability_365,
        has_availability,
        CAST(last_scraped AS DATE) AS last_scraped,
        CAST(calendar_last_scraped AS DATE) AS calendar_last_scraped,
        flag_price_missing,
        flag_price_invalid,
        flag_calendar_unknown,
        flag_calendar_inconsistent
    FROM staging
""")

saved_count = con.execute("SELECT count(*) FROM published_listings").fetchone()[0]
check("Количество после чтения S3", saved_count, staging_count)
output_head = s3.head_object(Bucket="mart", Key=mart_key)
show_table(["Объект", "Размер, байт"], [(MART_PATH, output_head["ContentLength"])])

Объект,"Размер, байт"
s3://mart/airbnb_amsterdam_listings/source_ingested_on=2026-10-23/listings.parquet,138672


<a id="checks"></a>
## 5. Проверки сохраненной витрины

Все запросы к `published_listings` ниже читают записанный Parquet на S3.
Ожидания следуют из модели и входного staging. При нарушении условия
`assert` останавливает выполнение. Сводка в конце показывает фактическое
и ожидаемое значение каждой проверки.

### Количество и схема

Все строки staging сохраняются. Проверяем нулевое число исключений,
баланс входных и сохраненных строк, а также имена, порядок и типы двенадцати полей.

In [6]:
row_balance = show_sql("""
    SELECT
        (SELECT count(*) FROM staging) AS staging_rows,
        count(*) AS saved_mart_rows,
        (SELECT count(*) FROM staging)
            - (SELECT count(*) FROM listings) AS excluded_rows
    FROM published_listings
""", ['Строк staging', 'Прочитано из mart', 'Исключено'])[0]

check("Баланс сохраненного файла", row_balance[0], row_balance[1] + row_balance[2])
check("Строки сохраненного файла", row_balance[1], staging_count)

expected_schema = [
    ("id", "VARCHAR"),
    ("district", "VARCHAR"),
    ("room_type", "VARCHAR"),
    ("price_eur", "DECIMAL(14,2)"),
    ("availability_365", "INTEGER"),
    ("has_availability", "BOOLEAN"),
    ("last_scraped", "DATE"),
    ("calendar_last_scraped", "DATE"),
    ("flag_price_missing", "BOOLEAN"),
    ("flag_price_invalid", "BOOLEAN"),
    ("flag_calendar_unknown", "BOOLEAN"),
    ("flag_calendar_inconsistent", "BOOLEAN"),
]

schema_rows = con.execute("DESCRIBE published_listings").fetchall()
actual_schema = [(row[0], row[1]) for row in schema_rows]
check("Имена, порядок и типы двенадцати полей", actual_schema, expected_schema)


show_table(["Поле", "Тип"], actual_schema)

Строк staging,Прочитано из mart,Исключено
10369,10369,0


Поле,Тип
id,VARCHAR
district,VARCHAR
room_type,VARCHAR
price_eur,"DECIMAL(14,2)"
availability_365,INTEGER
has_availability,BOOLEAN
last_scraped,DATE
calendar_last_scraped,DATE
flag_price_missing,BOOLEAN
flag_price_invalid,BOOLEAN


### Ключ, обязательные поля и меры

Ключ `id` должен быть заполнен и уникален внутри снимка. Проверяем его пропуски
и повторы, заполненность района, типа жилья, дат сбора и четырех флагов качества.
Без этих полей теряется смысл строки или необходимого среза.

Для цены и календарного признака допускаем исходные пропуски. Их количество
сравниваем со staging в следующих проверках. Доступность должна быть заполнена
в каждой строке. Проверяем допустимые категории типа жилья.

In [7]:
required_checks = con.execute("""
    SELECT
        count(*) FILTER (
            WHERE id IS NULL OR trim(id) = ''
        ) AS missing_id,
        count(id) - count(DISTINCT id) AS duplicate_id,
        count(*) FILTER (
            WHERE district IS NULL OR trim(district) = ''
        ) AS missing_district,
        count(*) FILTER (
            WHERE room_type IS NULL OR trim(room_type) = ''
        ) AS missing_room_type,
        count(*) FILTER (
            WHERE last_scraped IS NULL
               OR calendar_last_scraped IS NULL
        ) AS missing_dates,
        count(*) FILTER (
            WHERE flag_price_missing IS NULL
               OR flag_price_invalid IS NULL
               OR flag_calendar_unknown IS NULL
               OR flag_calendar_inconsistent IS NULL
        ) AS missing_quality_flags
    FROM published_listings
""").fetchone()

for label, value in zip(
    ["Пропуски ключа mart", "Повторы ключа mart", "Пропуски района mart",
     "Пропуски типа жилья mart", "Пропуски дат mart", "Пропуски флагов mart"],
    required_checks,
    strict=True,
):
    check(label, value, 0)


field_names = [row[0] for row in con.execute("DESCRIBE published_listings").fetchall()]
null_expressions = ", ".join(
    f"count(*) FILTER (WHERE {name} IS NULL)" for name in field_names
)
null_counts = con.execute(
    f"SELECT {null_expressions} FROM published_listings"
).fetchone()
source_expressions = ", ".join(
    f"count(*) FILTER (WHERE {('neighbourhood_cleansed' if name == 'district' else name)} IS NULL)"
    for name in field_names
)
source_null_counts = con.execute(
    f"SELECT {source_expressions} FROM staging"
).fetchone()
show_table(["Поле", "NULL", "Ожидается"], [
    (name, actual, expected)
    for name, actual, expected in zip(
        field_names, null_counts, source_null_counts, strict=True
    )
])
show_table(["Проверка", "Получено", "Ожидается"], [
    ("Повторы id", required_checks[1], 0)
])


bad_room_types = show_sql("""
    SELECT count(*)
    FROM published_listings
    WHERE room_type NOT IN (
        'Entire home/apt', 'Private room', 'Shared room', 'Hotel room'
    )
""", ["Недопустимых типов жилья"])[0][0]
check("Недопустимые типы жилья", bad_room_types, 0)

Поле,NULL,Ожидается
id,0,0
district,0,0
room_type,0,0
price_eur,3992,3992
availability_365,0,0
has_availability,136,136
last_scraped,0,0
calendar_last_scraped,0,0
flag_price_missing,0,0
flag_price_invalid,0,0


Проверка,Получено,Ожидается
Повторы id,0,0


Недопустимых типов жилья
0


### Цена и доступность: сохранение значений и правила расчета

Сравниваем со staging число пропусков, минимум и максимум цены и доступности.
Все исходные значения должны сохраниться. Заполненная цена должна быть
положительной, доступность должна находиться в диапазоне от 0 до 365 дней.
Нулевая доступность допустима. Пропуски цены не заменяются нулем.

Для основной медианы цены используются положительные заполненные цены без
флагов отсутствующей или некорректной цены. Для основной медианы доступности
требуется `has_availability = true`. Ограничения относятся к будущему расчету,
соответствующие объявления остаются в витрине.

In [8]:
measure_stats = con.execute("""
    SELECT
        'staging' AS dataset,
        count(*) FILTER (WHERE price_eur IS NULL) AS null_prices,
        min(price_eur) AS min_price,
        max(price_eur) AS max_price,
        count(*) FILTER (
            WHERE availability_365 IS NULL
        ) AS null_availability,
        min(availability_365) AS min_available_days,
        max(availability_365) AS max_available_days,
        count(*) FILTER (
            WHERE has_availability IS NULL
        ) AS null_calendar_flags
    FROM staging

    UNION ALL

    SELECT
        'mart' AS dataset,
        count(*) FILTER (WHERE price_eur IS NULL),
        min(price_eur),
        max(price_eur),
        count(*) FILTER (WHERE availability_365 IS NULL),
        min(availability_365),
        max(availability_365),
        count(*) FILTER (WHERE has_availability IS NULL)
    FROM published_listings

    ORDER BY dataset
""").fetchall()

check("Профиль мер и пропусков относительно staging",
      measure_stats[0][1:], measure_stats[1][1:])


show_table(["Слой", "NULL цены", "Мин. цена, евро", "Макс. цена, евро"], [
    ("Mart" if row[0] == "mart" else "Staging", *row[1:4])
    for row in measure_stats
])
show_table(["Слой", "NULL доступности", "Мин., дней", "Макс., дней",
            "Неизвестный календарь"], [
    ("Mart" if row[0] == "mart" else "Staging", *row[4:])
    for row in measure_stats
])
measure_checks = show_sql("""
    SELECT
        count(*) FILTER (WHERE price_eur IS NOT NULL AND price_eur <= 0),
        count(*) FILTER (
            WHERE availability_365 IS NULL OR availability_365 NOT BETWEEN 0 AND 365
        )
    FROM published_listings
""", ["Неположительных цен", "Некорректной доступности"])[0]
check("Неположительные цены", measure_checks[0], 0)
check("Некорректная доступность", measure_checks[1], 0)

Слой,NULL цены,"Мин. цена, евро","Макс. цена, евро"
Mart,3992,14.70,11412.00
Staging,3992,14.70,11412.00


Слой,NULL доступности,"Мин., дней","Макс., дней",Неизвестный календарь
Mart,0,0,365,136
Staging,0,0,365,136


Неположительных цен,Некорректной доступности
0,0


### Даты сбора и флаги качества

Проверяем период сбора объявлений и календарей с 15 по 24 июня 2026.
Даты после приведения к DATE должны соответствовать исходным значениям.
Количество строк с каждым флагом качества сверяется со staging.

Флаг календарного несоответствия остается диагностическим признаком.
Он не доказывает ошибку источника и не является причиной удаления объявления.

In [9]:
dates_outside_period = con.execute("""
    SELECT count(*) FILTER (
        WHERE last_scraped NOT BETWEEN DATE '2026-06-15' AND DATE '2026-06-24'
           OR calendar_last_scraped NOT BETWEEN DATE '2026-06-15' AND DATE '2026-06-24'
    )
    FROM published_listings
""").fetchone()[0]
check("Даты вне периода", dates_outside_period, 0)


flag_counts = show_sql("""
    SELECT
        'staging' AS dataset,
        count(*) FILTER (WHERE flag_price_missing) AS missing_prices,
        count(*) FILTER (WHERE flag_price_invalid) AS invalid_prices,
        count(*) FILTER (WHERE flag_calendar_unknown) AS unknown_calendars,
        count(*) FILTER (
            WHERE flag_calendar_inconsistent
        ) AS inconsistent_calendars
    FROM staging

    UNION ALL

    SELECT
        'mart' AS dataset,
        count(*) FILTER (WHERE flag_price_missing),
        count(*) FILTER (WHERE flag_price_invalid),
        count(*) FILTER (WHERE flag_calendar_unknown),
        count(*) FILTER (WHERE flag_calendar_inconsistent)
    FROM published_listings

    ORDER BY dataset
""", [
    'Слой',
    'Без цены с флагом',
    'Некорректных цен с флагом',
    'Неизвестный календарь',
    'Календарное несоответствие',
])

check("Количество каждого флага относительно staging",
      flag_counts[0][1:], flag_counts[1][1:])

Слой,Без цены с флагом,Некорректных цен с флагом,Неизвестный календарь,Календарное несоответствие
mart,3992,0,136,151
staging,3992,0,136,151


### Сохранение значений и кратности строк

Сверяем все двенадцать полей записанной витрины с ожидаемым преобразованием
всех строк staging через EXCEPT ALL в обе стороны. Проверка учитывает
значения NULL и кратность строк. Она подтверждает точный перенос ключа,
срезов, мер, дат после преобразования и четырех флагов качества,
включая значения в помеченных строках.

In [10]:
exact_checks = show_sql("""
    SELECT
        (
            SELECT count(*)
            FROM (
                SELECT * FROM listings
                EXCEPT ALL
                SELECT * FROM published_listings
            ) AS expected_only
        ) AS missing_or_changed_rows,
        (
            SELECT count(*)
            FROM (
                SELECT * FROM published_listings
                EXCEPT ALL
                SELECT * FROM listings
            ) AS saved_only
        ) AS unexpected_or_changed_rows
""", ['Отсутствуют или изменены в mart', 'Лишние или изменены в mart'])[0]

check("Отсутствующие или измененные строки", exact_checks[0], 0)
check("Лишние или измененные строки", exact_checks[1], 0)

show_sql("""
    SELECT
        id,
        price_eur,
        flag_price_missing,
        has_availability,
        availability_365,
        flag_calendar_unknown,
        flag_calendar_inconsistent
    FROM published_listings
    WHERE flag_price_missing
       OR flag_calendar_unknown
       OR flag_calendar_inconsistent
    ORDER BY flag_calendar_inconsistent DESC, id
    LIMIT 5
""", [
    'id',
    'Цена, евро за ночь',
    'Флаг отсутствующей цены',
    'Наличие календаря',
    'Доступность, дней',
    'Флаг неизвестного календаря',
    'Флаг несоответствия календаря',
])

print("Значения, флаги и кратность строк: проверки пройдены.")

Отсутствуют или изменены в mart,Лишние или изменены в mart
0,0


id,"Цена, евро за ночь",Флаг отсутствующей цены,Наличие календаря,"Доступность, дней",Флаг неизвестного календаря,Флаг несоответствия календаря
1034672457625987114,143.80,False,False,29,False,True
1080400468025953047,62.81,False,False,215,False,True
1080400470981051445,62.81,False,False,296,False,True
1080404361174435888,90.91,False,False,298,False,True
1091815580643187209,77.69,False,False,88,False,True


Значения, флаги и кратность строк: проверки пройдены.


### Пригодность для выбранного вопроса

Контрольный запрос группирует объявления по району и типу жилья,
считает все объявления и число пригодных записей для каждой будущей медианы.
Показываются первые десять групп.

Запрос подтверждает наличие нужных срезов и возможность применять
правила отбора значений отдельно для цены и календарной доступности.
Полные метрики, рейтинги районов и содержательные выводы здесь не рассчитываются.

Дополнительно проверяется, что сумма количества объявлений по всем группам
равна числу строк витрины.

In [11]:
group_counts = con.execute("""
    SELECT
        district,
        room_type,
        count(*) AS listings_count,
        count(*) FILTER (
            WHERE price_eur > 0
              AND NOT flag_price_missing
              AND NOT flag_price_invalid
        ) AS valid_price_count,
        count(*) FILTER (
            WHERE has_availability = true
              AND availability_365 BETWEEN 0 AND 365
        ) AS valid_availability_count
    FROM published_listings
    GROUP BY district, room_type
    ORDER BY district, room_type
""").fetchall()

group_balance = (len(group_counts), sum(row[2] for row in group_counts))
check("Баланс контрольной группировки", group_balance[1], saved_count)
show_table(
    ["Район", "Тип жилья", "Объявлений", "Пригодных цен", "Пригодных календарей"],
    group_counts[:10],
)
show_table(["Всего групп", "Сумма объявлений по всем группам"], [group_balance])
print("Пригодность для выбранного вопроса: проверка пройдена.")


Район,Тип жилья,Объявлений,Пригодных цен,Пригодных календарей
Bijlmer-Centrum,Entire home/apt,26,15,26
Bijlmer-Centrum,Hotel room,10,10,10
Bijlmer-Centrum,Private room,26,22,22
Bijlmer-Oost,Entire home/apt,28,13,27
Bijlmer-Oost,Private room,11,11,11
Bos en Lommer,Entire home/apt,507,228,500
Bos en Lommer,Private room,39,29,39
Buitenveldert - Zuidas,Entire home/apt,87,56,86
Buitenveldert - Zuidas,Private room,32,31,32
Centrum-Oost,Entire home/apt,614,399,606


Всего групп,Сумма объявлений по всем группам
55,10369


Пригодность для выбранного вопроса: проверка пройдена.


### Контрольная сводка

Дополнительно проверяем, что метаданные входного объекта не изменились за время
выполнения. Сводка показывает полученное и ожидаемое значение каждой проверки.
Все проверки относятся к выбранному снимку данных.

In [12]:
source_after = s3.head_object(Bucket="staging", Key=staging_key)
source_metadata_fields = ("ETag", "ContentLength", "LastModified")
check(
    "Входной объект не изменен",
    tuple(source_after[field] for field in source_metadata_fields),
    tuple(source_before[field] for field in source_metadata_fields),
)
show_table(["Проверка", "Получено", "Ожидается", "Итог"], [
    (label, actual, expected, "Пройдена" if passed else "Нарушение")
    for label, actual, expected, passed in check_results
])
run_summary = {
    "staging_rows": staging_count,
    "mart_rows": saved_count,
    "excluded_rows": balance[2],
    "schema": actual_schema,
    "flag_counts": flag_counts[0][1:],
    "content_difference": exact_checks,
    "groups_count": group_balance[0],
    "checks_total": len(check_results),
    "checks_passed": sum(passed for _, _, _, passed in check_results),
}
show_table(["Проверок", "Пройдено", "Строк mart", "Исключено"], [
    (run_summary["checks_total"], run_summary["checks_passed"], saved_count, balance[2])
])
writer.close()
con.close()

Проверка,Получено,Ожидается,Итог
Наличие нужных полей staging,set(),set(),Пройдена
Пропуски ключа на входе,0,0,Пройдена
Повторы ключа на входе,0,0,Пройдена
Пропуски района на входе,0,0,Пройдена
Пропуски типа жилья на входе,0,0,Пройдена
Некорректные даты объявления на входе,0,0,Пройдена
Некорректные даты календаря на входе,0,0,Пройдена
Баланс при преобразовании,10369,10369,Пройдена
Сохранение строк при преобразовании,10369,10369,Пройдена
Исключенные строки,0,0,Пройдена


Проверок,Пройдено,Строк mart,Исключено
30,30,10369,0


<a id="result"></a>
## 6. Оформите результат

Модель, происхождение всех выходных полей, краткий итог проверок и ограничения
описаны в [docs/mart.md](../docs/mart.md). Витрина поддерживает выбранный
основной вопрос Project Brief о различиях цен и календарной доступности
между районами при сравнении одинаковых типов жилья.

Итог сборки: все 10 369 строк staging сохранены в витрине, исключенных строк нет.
Гранулярность и кратность строк сохранены. В выходном файле 12 полей,
ключ `id` заполнен и уникален. Перенесены 3 992 флага отсутствующей цены,
0 флагов некорректной цены, 136 флагов неизвестного календарного признака
и 151 флаг календарного несоответствия. Все 30 проверок сборки пройдены.

Витрина сохранена и прочитана из S3:

```text
s3://mart/airbnb_amsterdam_listings/source_ingested_on=2026-10-23/listings.parquet
```

Для расчета показателей зафиксированы отдельные правила цены и доступности.
Объявления с пропусками цены или календарного признака остаются в витрине.
Медиана цены рассчитывается по положительным заполненным ценам без флагов
отсутствующей или некорректной цены. Основная медиана доступности рассчитывается
при `has_availability = true`, включая нулевую доступность. Эти условия
не применяются как фильтр строк при сборке витрины.

Ограничения: отсутствие цены примерно у 38,5% объявлений может смещать сравнения.
Календарная недоступность не различает бронирование и блокировку хозяином.
Малые группы и разные даты сбора ограничивают сопоставимость.
Используется один снимок, история изменений объявлений отсутствует.
В этой лабораторной выполнены сборка витрины и проверки ее пригодности
для выбранного вопроса.

Ноутбук сохранен с актуальными SQL-запросами и результатами выполнения.
Подробные проверки и таблицы результатов находятся в [разделе 5](#checks),
в описании модели оставлен краткий итог.